# Base Modeling

Este notebook parte del **dataset final** de Data Preparation (`dataset_final.parquet`, guardado en la última celda de `DataPreparation.ipynb`).

Aquí se hacen la normalización de los datos y la definición de los folds de validación con `TimeSeriesSplit`.

### 1. Conectar con Google Drive e importar librerías

In [10]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!rm -rf vms

In [12]:
import sys
!git clone https://github.com/kamimartinez/VMS-prediction.git vms
def change_branch(branch):
  !git -C vms checkout {branch}
change_branch("feat/lstm")
sys.path.append(os.path.abspath("vms/notebooks"))

Cloning into 'vms'...
remote: Enumerating objects: 365, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 365 (delta 71), reused 93 (delta 44), pack-reused 230 (from 1)
Receiving objects: 100% (365/365), 16.65 MiB | 21.24 MiB/s, done.
Resolving deltas: 100% (149/149), done.
branch 'feat/lstm' set up to track 'origin/feat/lstm'.
Switched to a new branch 'feat/lstm'


In [13]:
!ls vms/notebooks

data  evaluation  modeling


### 2. Cargar el dataset final
Se lee desde `Datos/Procesado`. Si no existe, primero hay que ejecutar `DataPreparation.ipynb` completo.

In [14]:
RUTA_DATOS = "/content/drive/My Drive/Datos/Procesado"

def load_dataset(file="dataset_final_serie_tiempo.parquet"):
  dataset_final = pd.read_parquet(os.path.join(RUTA_DATOS, file))
  dataset_final["bloque_tiempo"] = pd.to_datetime(dataset_final["bloque_tiempo"])
  dataset_final = dataset_final.sort_values(["bloque_tiempo", "ms"]).reset_index(drop=True)

  print(f"dataset_final: {len(dataset_final):,} bloques x {dataset_final.shape[1]} columnas")
  print(list(dataset_final.columns))
  return dataset_final

### 3. Parámetros

In [15]:
TARGET = "pct_posible_cola"
N_FOLDS = 5
tscv = TimeSeriesSplit(n_splits=N_FOLDS)

# Normalización de datos

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd

TARGET = "pct_posible_cola"

datasets = ["dataset_final_regresion.parquet", "dataset_final_serie_tiempo.parquet"]

df_final = load_dataset(datasets[1])
df_model = df_final.copy()

df_model = pd.get_dummies(df_model, columns=["ms", "temporada"], dtype="float32")

columnas_excluir = [TARGET, "bloque_tiempo"]

X = df_model.drop(columns=columnas_excluir)
y = df_model[TARGET]

categoricas = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numericas = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
    ],
    remainder="drop",
    verbose_feature_names_out=False,   # quita los prefijos num__ / cat__
)

# Que devuelva un DataFrame directamente (sklearn >= 1.2)
preprocessor.set_output(transform="pandas")

X_normalizado = preprocessor.fit_transform(X)

dataset_final: 179,850 bloques x 24 columnas
['ms', 'bloque_tiempo', 'total_ordenios', 'produccion_promedio', 'duracion_total_min', 'patadas_totales', 'desviaciones_leche_totales', 'ordenios_con_lavado', 'pct_primer_parto', 'pct_fresca', 'pct_alta', 'pct_baja', 'pct_extendida', 'hubo_ordenios', 'pct_ocupacion_robot', 'hubo_lavado', 'ordenios_otros_robots', 'hora', 'dia_semana', 'temporada', 'min_desde_alimento', 'vacas_activas_dia', 'sin_etapa', 'pct_posible_cola']


# Separación de train y tests

In [17]:
no_normalizadas = df_model[columnas_excluir]   # TARGET, ms, bloque_tiempo

# Reintegrar por índice
df_final = pd.concat([no_normalizadas, X_normalizado], axis=1)

def split_df(df_final, target_col=TARGET, shuffle=False):
  df_train, df_temp = train_test_split(
      df_final, test_size=0.3, shuffle=shuffle
  )
  df_val, df_test = train_test_split(df_temp, test_size=0.5, shuffle=shuffle)

  df_x_train, df_y_train = df_train.drop(columns=[target_col]), df_train[target_col]
  df_x_val,df_y_val = df_val.drop(columns=[target_col]), df_val[target_col]
  df_x_test, df_y_test = df_test.drop(columns=[target_col]), df_test[target_col]

  return (df_x_train, df_y_train), (df_x_val, df_y_val), (df_x_test, df_y_test)

> **Nota:** en esta celda el `preprocessor` se ajusta con todo el dataset, lo que deja ver información del periodo de prueba. Para entrenar y evaluar modelos se debe ajustar (`fit`) solo con los bloques de entrenamiento de cada fold y aplicar `transform` a los de prueba.

# Validación

Se usa el mismo `TimeSeriesSplit` de Data Preparation (5 folds), ahora sobre `dataset_final`: cada fold entrena con el pasado y prueba con el periodo siguiente.

In [19]:
df_train, df_val, df_test = split_df(df_final.drop(columns="bloque_tiempo"))
# df_train = df_train.sort_values(["bloque_tiempo", "ms"]).reset_index(drop=True)

# folds = []
# for i, (idx_ent, idx_prueba) in enumerate(tscv.split(df_train), start=1):
#     t = df_train["bloque_tiempo"]
#     folds.append({
#         "fold": i,
#         "entrenamiento": f"{t.iloc[idx_ent[0]].date()} a {t.iloc[idx_ent[-1]].date()}",
#         "prueba": f"{t.iloc[idx_prueba[0]].date()} a {t.iloc[idx_prueba[-1]].date()}",
#         "bloques_entrenamiento": len(idx_ent),
#         "bloques_prueba": len(idx_prueba),
#     })
# display(pd.DataFrame(folds))


In [22]:
from modeling.model_base import BaseModel
from modeling.lstm_model import LSTMModel
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from pathlib import Path

df_train_x, df_train_y = df_train
df_val_x, df_val_y = df_val

model_path = Path.cwd()

model: BaseModel = LSTMModel()
X = model.prepare(df_train_x.values)
X_val = model.prepare(df_val_x.values)
print(X.shape)

model = model.compile(input_shape=X.shape[1:])
model.summary()

(125895, 27, 1)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 27, 128)        │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 27, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 198,273 (774.50 KB)

 Trainable params: 198,273 (774.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    ModelCheckpoint(
        model_path / "lstm_model.keras", monitor="val_loss", save_best_only=True
    ),
]

history = model.fit(
    X, df_train_y.values,
    validation_data=(X_val, df_val_y.values),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    verbose=1,
)

Epoch 1/50
2088/3935 ━━━━━━━━━━━━━━━━━━━━ 2:22 77ms/step - accuracy: 0.6927 - loss: nan - precision: 0.0000e+00

KeyboardInterrupt: 